In [1]:
import sys
import os

project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import tz_pypsa
import pandas as pd
import numpy as np
from tz_pypsa.model import Model
import tz_pypsa.wrangle as wrangle                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     

In [2]:
n=Model.load_csv_from_dir('./stock_models/Taiwan',
                          2024,
                          frequency='1h',
                          timesteps=8760,
                          backstop=False)     

INFO:pypsa.networks:Repeating time-series for each investment period and converting snapshots to a pandas.MultiIndex.
INFO:pypsa.io:Imported network Taiwan has buses, carriers, generators, loads, storage_units


In [ ]:
# solve the model
n.optimize()

In [ ]:
# network.export_to_netcdf('/home/yantidiah/tza-pypsa/output/calibration/apg_cferelated-1h_5Mar_11PM_8bus.nc')

# CHECKING RESULTS

In [ ]:
# --- CHECKING BACKSTOP --- #

network.generators_t.p.filter(like='Backstop').sum()

In [ ]:
# --- CHECKING ENERGY BALANCE --- #

import tz_pypsa
fig = tz_pypsa.plotting.energy_balance(network,'2023')
fig.update_layout(width=1000,height=500)
fig.show()

In [ ]:
# --- CHECKING GENERATION PER TYPE --- #

import plotly.express as px

loads = (
    network
    .loads_t
    .p_set
    .resample('YE')
    .sum()
    .div(1e6)
    .melt()
    .sort_values(by='Load')
    .reset_index(drop=True)
)

total_generation = (
    network
    .generators_t
    .p
    # .loc[2023]
    .resample('YE')
    .sum()
    .groupby([network.generators.bus, network.generators.carrier], axis=1)
    .sum()
    .div(1e6)
    .melt()
    .sort_values(by='bus')
)

# define order for x-axis
cat_order = total_generation.sort_values(by='bus').bus.unique().tolist()


fig = px.bar(
    total_generation, 
    x="bus", 
    y="value", 
    color = "carrier",
    category_orders={'bus' : cat_order},
    #barmode="group",
)

fig.add_scatter(
    x=loads.Load,
    y=loads.value,
    mode='markers',
    name='Load',
    marker=dict(
        size=7,
        color='red',
        symbol='circle',
    ),
)

fig.update_layout(
        yaxis_title= 'Generation (TWh)',
        xaxis_title='',
        # title='Installed capacity in ' + str(2023),
        width=1000,
        height=500,
        xaxis_tickangle=-45
    )

fig

In [ ]:
# --- CHECK GENERATION MIX --- #

from plotly.subplots import make_subplots
import plotly.graph_objects as go


generation = (
    network
    .generators_t
    .p
    .resample('YE')
    .sum()
    .groupby([network.generators.bus, network.generators.carrier], axis=1)
    .sum()
    .melt()
    .sort_values(by='bus')
    )

# change empty technology type into 'backstop'
generation.carrier.replace('', 'backstop', inplace=True)

# Apply scaling factor
generation.value = generation.value.div(1e6)

# Generate pie chart for each region
fig = make_subplots(rows=1, cols=3, specs=[[{"type": "pie"},{"type": "pie"},{"type": "pie"}]],subplot_titles=('IDNKA','MYSSK','MYSSH'))
color_map = network.carriers.color.to_dict()

fig.add_trace(go.Pie(
     values=generation.loc[generation.bus == 'IDNKA'].value,labels=generation.carrier.loc[generation.bus == 'IDNKA'],
     domain=dict(x=[0, 0.3]),name="IDNKA"),1,1)

fig.add_trace(go.Pie(
     values=generation.loc[generation.bus == 'MYSSK'].value,labels=generation.carrier.loc[generation.bus == 'MYSSK'],
     domain=dict(x=[0.3, 0.6]),name="MYSSK"),1,2)

fig.add_trace(go.Pie(
     values=generation.loc[generation.bus == 'MYSSH'].value,labels=generation.carrier.loc[generation.bus == 'MYSSH'],
     domain=dict(x=[0.6, 0.9]),name="MYSSH"),1,3)

fig.update_traces(textposition='inside')
fig.update_layout(title_text="Generation mix in IDNKA, MYSSK and MYSSH", width=1000,height=500)

fig.show()


In [ ]:
# --- CALIBRATION 1b: GENERATION 2023- Official data vs model (all Indochina) --- #

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Extract country data and results
country_list = [
                # 'LAO',
                # 'KHM',
                # 'MMR',
                'MYS',
                'SGP',
                'THA',
                # 'VNM'
                ]
fig = make_subplots(rows=len(country_list), cols=1,shared_xaxes=False, subplot_titles=country_list)
for country in country_list:

    # Pypsa results
    gen_pyspsa = (
        network
        .generators_t
        .p
        # .loc[2023]
        .filter(like=country)
        .groupby(by=[network.generators.carrier],axis=1)
        .sum().sum().div(1e6)
        .to_frame().reset_index().rename(columns={'carrier':'Fuel',0:'gen_pypsa'})
    )

    # Extract official data for Singapore and Malaysia
    gen_ofc_doc = pd.read_excel('/home/yantidiah/tza-pypsa/Generation_2023_official_v2.xlsx')
    gen_ofc = (
        gen_ofc_doc[gen_ofc_doc['Country_code'] == country]
        .reset_index().rename(columns={'Generation_GWh':'gen_ofc'})
    )
    # Combine both
    gen = gen_pyspsa.merge(gen_ofc,how='outer',on='Fuel')

    # subplot
    fig.add_trace(go.Bar(x=gen['Fuel'], y=gen['gen_ofc'].div(1e3),name=country+' ref'),country_list.index(country)+1,1)
    fig.add_trace(go.Bar(x=gen['Fuel'], y=gen['gen_pypsa'],name=country+' results'),country_list.index(country)+1,1)

fig.update_layout(title_text="2023 GENERATION (TWh) COMPARISON",
                  width=1000,height=800)

fig.show()

In [ ]:
# --- CALIBRATION 1c: GENERATION 2023 COMPARISON (%)- Official data vs model (all Indochina) --- #

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Extract country data and results
country_list = [
    # 'KHM',
    # 'LAO',
    'MYS',
    'SGP',
    'THA',
    # 'VNM'
    ]
fig = make_subplots(rows=len(country_list), cols=1,shared_xaxes=False, subplot_titles=country_list)
for country in country_list:

    # Pypsa results
    gen_pyspsa = (
        network
        .generators_t
        .p
        # .loc[2023]
        .filter(like=country)
        .groupby(by=[network.generators.carrier],axis=1)
        .sum().sum().div(1e6)
        .to_frame().reset_index().rename(columns={'carrier':'Fuel',0:'gen_pypsa'})
    )

    # Extract official data for Singapore and Malaysia
    gen_ofc_doc = pd.read_excel('/home/yantidiah/tza-pypsa/Generation_2023_official_v2.xlsx')
    gen_ofc = (
        gen_ofc_doc[gen_ofc_doc['Country_code'] == country]
        .reset_index().rename(columns={'Generation_GWh':'gen_ofc'})
    )
    # Combine both
    gen = gen_pyspsa.merge(gen_ofc,how='outer',on='Fuel')

    # subplot
    fig.add_trace(go.Bar(x=gen['Fuel'], y=(gen['gen_ofc'].div(1e3)-gen['gen_pypsa'])/gen['gen_ofc'].div(1e3),showlegend=False),country_list.index(country)+1,1)
    
    fig.add_hline(y=0.1)
    fig.add_hline(y=-0.1)


fig.update_layout(title_text="2023 GENERATION COMPARISON (%)",
                  width=800,height=800)

fig.show()

In [ ]:
# --- CALIBRATION 2: CAPACITY FACTOR 2023 - Official data vs model (only MYS and SGP) --- #

# Extract CF results for Singapore and Malaysia
cf_pypsa = (network.statistics.capacity_factor(groupby=['bus','carrier'])
            .to_frame().reset_index().rename(columns={'carrier':'Fuel',0:'cf_pypsa'})
)
cf_pypsa['Fuel'] = cf_pypsa['Fuel'].str.lower()

# cf_pypsa['Generator'].loc[['SGPXX','MYSPE']]

# Official doc results
cap_sgp = (
    network.generators.p_nom_opt
    .filter(like='SGP')
    .groupby(by=[network.generators.carrier])
    .sum().div(1e3)
    .to_frame().reset_index().rename(columns={'carrier':'Fuel',0:'cap'})
)
sgp_cf = sgp_gen.merge(cap_sgp,how='outer',on='Fuel')
sgp_cf['cf_ofc'] = sgp_cf['gen_ofc'].div(sgp_cf['p_nom_opt'] * 8760)
sgp_cf = sgp_cf.merge(cf_pypsa.loc[(cf_pypsa['bus'] == 'SGPXX') & (cf_pypsa['component'] == 'Generator')],
                      how='outer',on='Fuel')

cap_mys = (
    network.generators.p_nom_opt
    .filter(like='MYS')
    .groupby(by=[network.generators.carrier])
    .sum().div(1e3)
    .to_frame().reset_index().rename(columns={'carrier':'Fuel',0:'cap'})
)
mys_cf = mys_gen.merge(cap_mys,how='outer',on='Fuel')
mys_cf['cf_ofc'] = mys_cf['gen_ofc'].div(mys_cf['p_nom_opt'] * 8760)
mys_cf = mys_cf.merge(cf_pypsa.loc[(cf_pypsa['bus'] == 'MYSPE') & (cf_pypsa['component'] == 'Generator')],
                      how='outer',on='Fuel')

# subplot for SGP and MYS
fig = go.Figure()
fig = make_subplots(rows=2, cols=1,shared_xaxes=False, subplot_titles=('Singapore','Malaysia'))

fig.add_trace(go.Bar(x=sgp_cf['Fuel'], y=sgp_cf['cf_ofc'],name='SGP ref'),1,1)
fig.add_trace(go.Bar(x=sgp_cf['Fuel'], y=sgp_cf['cf_pypsa'],name='SGP results'),1,1)

fig.add_trace(go.Bar(x=mys_cf['Fuel'], y=mys_cf['cf_ofc'],name='MYS ref'),2,1)
fig.add_trace(go.Bar(x=mys_cf['Fuel'], y=mys_cf['cf_pypsa'],name='MYS results'),2,1)

fig.update_layout(title_text="2023 CF COMPARISON", height=500, width=800)

fig.show()

In [ ]:
# --- CALIBRATION 3a: DISPATCH MONTHLY 2023 - Singapore (model run) --- #
# Note: this is the monthly averaged dispatch and demand profile for 2023

fig = tz_pypsa.plotting.dispatch(network,'2023',iso_code='SGP',resample='D', show_exports=True, show_imports=True)
fig.update_layout(title_text='SGP dispatch 2023 (model run)',width=1000,height=500)
fig.show()

In [ ]:
# --- CALIBRATION 3b: DISPATCH MONTHLY 2023 - Singapore (ofc) --- #
# Note: this is the monthly averaged dispatch and demand profile for 2023

sgp_dispatch_ofc = pd.read_csv('/home/yantidiah/tza-pypsa/SGP_hourly_demand_solargen_cost_2023-2024.csv')
sgp_dispatch_ofc['timestamp'] = pd.to_datetime(sgp_dispatch_ofc['timestamp'])
sgp_dispatch_ofc = sgp_dispatch_ofc[sgp_dispatch_ofc['timestamp'].dt.year == 2023]
sgp_dispatch_ofc.set_index('timestamp',inplace=True)

resample_sgp_dispatch_hourly = sgp_dispatch_ofc.resample('H').median(numeric_only=True)
resample_sgp_dispatch = resample_sgp_dispatch_hourly.resample('D').sum()

# plot daily dispatch
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=resample_sgp_dispatch.index, 
        y=(resample_sgp_dispatch['demand_mw'] - resample_sgp_dispatch['solar_mw']).div(1e3), 
        mode='lines',
        name='gas',
        line=dict(color='black'),
        stackgroup='one'
        )
)
fig.add_trace(
    go.Scatter(
        x=resample_sgp_dispatch.index, 
        y=(resample_sgp_dispatch['solar_mw']).div(1e3), 
        mode='lines',
        name='solar',
        line=dict(color='yellow'),
        stackgroup='one'
        )
)
fig.update_layout(
        xaxis=dict(
            rangeslider=dict(
                visible=True
            ),
            type="date",
        ),
        # yaxis_range=(0, (df.drop('timestep',axis=1).sum(axis=1).max() / 1e3)*1.05 ),
        yaxis_title=f'Generation (GW)',
        xaxis_title='Time (Day)',
        title='SGP daily dispatch 2023 (ofc)',
        width=1000,
        height=500,
    )

fig.show()

In [ ]:
# --- CALIBRATION 3c: DISPATCH MONTHLY 2023 - Malaysia (model run) --- #
# Note: this is the monthly averaged dispatch and demand profile for 2023

fig = tz_pypsa.plotting.dispatch(network,'2023',iso_code='MYSPE',resample='D', show_exports=True, show_imports=True)
fig.update_layout(title_text='MYS dispatch 2023 (model run)',width=1000,height=500)
fig.show()

In [ ]:
# --- CALIBRATION 3d: DISPATCH MONTHLY 2023 - Malaysia (ofc) --- #
# Note: this is the monthly averaged dispatch and demand profile for 2023

mys_dispatch_ofc = pd.read_csv('/home/yantidiah/tza-pypsa/bq_export_malaysia_gso.csv')
mys_dispatch_ofc['DT'] = pd.to_datetime(mys_dispatch_ofc['DT'])
mys_dispatch_ofc = mys_dispatch_ofc[mys_dispatch_ofc['DT'].dt.year == 2023]
mys_dispatch_ofc.set_index('DT',inplace=True)
mys_dispatch_ofc = mys_dispatch_ofc.drop(mys_dispatch_ofc.columns[-1],axis=1)

resample_mys_dispatch_hourly = mys_dispatch_ofc.resample('H').median()
resample_mys_dispatch = resample_mys_dispatch_hourly.resample('D').sum()

# plot daily dispatch
fig = go.Figure()

for tech in resample_mys_dispatch.columns:
    fig.add_trace(
        go.Scatter(
            x=resample_mys_dispatch.index, 
            y=resample_mys_dispatch[tech].div(1e3), 
            mode='lines',
            name=tech,
            stackgroup='one'
            )
    )

fig.update_layout(
        xaxis=dict(
            rangeslider=dict(
                visible=True
            ),
            type="date",
        ),
        # yaxis_range=(0, (df.drop('timestep',axis=1).sum(axis=1).max() / 1e3)*1.05 ),
        yaxis_title=f'Generation (GW)',
        xaxis_title='Time (Day)',
        title='MYS daily dispatch 2023 (ofc)',
        width=1000,
        height=500,
    )

fig.show()

In [ ]:
# --- CALIBRATION 3e: DISPATCH MONTHLY 2023 - Singapore (ofc total vs model per type) --- #

fig = tz_pypsa.plotting.dispatch(network,'2023',iso_code='SGP',resample='D', show_exports=True, show_imports=True)
fig.add_trace(
    go.Scatter(
        x=resample_sgp_dispatch.index, 
        y=resample_sgp_dispatch['demand_mw'].div(1e3), 
        mode='lines',
        name='demand_official',
        line=dict(color='red'),
        # stackgroup='one'
        )
)
fig.update_layout(title_text='SGP dispatch 2023 (model run vs total dispatched demand)',width=1000,height=500)
fig.show()

In [ ]:
# --- CALIBRATION 3f: DISPATCH MONTHLY 2023 - Malaysia (ofc total vs model per type) --- #

fig = tz_pypsa.plotting.dispatch(network,'2023',iso_code='MYSPE',resample='D', show_exports=True, show_imports=True)
fig.add_trace(
    go.Scatter(
        x=resample_mys_dispatch.index, 
        y=((resample_mys_dispatch['Coal']+
           resample_mys_dispatch['Gas']+
           resample_mys_dispatch['CoGen']+
           resample_mys_dispatch['Oil']+
           resample_mys_dispatch['Hydro']+
           resample_mys_dispatch['Solar'])
           .div(1e3)), 
        mode='lines',
        name='demand_official',
        line=dict(color='red'),
        # stackgroup='one'
        )
)
fig.update_layout(title_text='MYS dispatch 2023 (model run vs total dispatched generation)',width=1000,height=500)
fig.show()

In [ ]:
# --- Calibration 5a: Monthly comparison of CF 2023 (only MYSPE) --- #

# Monthly official
resample_mys_dispatch_monthly = (resample_mys_dispatch_hourly
                                 .resample('M')
                                 .sum()
                                 .div(1e6) # MWh to TWh
                                )

# Monthly model
monthly_gen_pyspsa = (
    network
    .generators_t
    .p
    .filter(like='MYSPE')
    .groupby(by=[network.generators.carrier],axis=1)
    .sum().div(1e6) # MWh to TWh
    .resample('M')
    .sum()
    .reset_index()
    .rename(columns={'index':'timestamp'})
)

fig = make_subplots(rows=4, cols=1,shared_xaxes=False, subplot_titles = ['Coal','Gas','Hydro','Solar'])
month = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

# subplot 1 - coal generation comparison
fig.add_trace(go.Bar(x=month, y=resample_mys_dispatch_monthly['Coal']*0.89, name="reference",legendgroup="reference",marker=dict(color='blue')),1,1)
fig.add_trace(go.Bar(x=month, y=monthly_gen_pyspsa['coal'], name="model",legendgroup="model",marker=dict(color='red')),1,1)

# subplot 2 - gas generation comparison
fig.add_trace(go.Bar(x=month, y=resample_mys_dispatch_monthly['Gas']*0.89,legendgroup="reference",showlegend=False,marker=dict(color='blue')),2,1)
fig.add_trace(go.Bar(x=month, y=monthly_gen_pyspsa['gas'],legendgroup="model",showlegend=False,marker=dict(color='red')),2,1)

# subplot 3 - hydro generation comparison
fig.add_trace(go.Bar(x=month, y=resample_mys_dispatch_monthly['Hydro']*0.89,legendgroup="reference",showlegend=False,marker=dict(color='blue')),3,1)
fig.add_trace(go.Bar(x=month, y=monthly_gen_pyspsa['hydro'],legendgroup="model",showlegend=False,marker=dict(color='red')),3,1)

# subplot 4 - solar generation comparison
fig.add_trace(go.Bar(x=month, y=resample_mys_dispatch_monthly['Solar'],legendgroup="reference",showlegend=False,marker=dict(color='blue')),4,1)
fig.add_trace(go.Bar(x=month, y=monthly_gen_pyspsa['solar'],legendgroup="model",showlegend=False,marker=dict(color='red')),4,1)

fig.update_layout(title_text="MYSPE - 2023 GENERATION (TWh) COMPARISON",
                  width=800,height=1000)

fig.show()



In [ ]:
# --- Calibration 5b: Monthly comparison of CF 2023 (only MYSPE) --- #

# Monthly official
resample_mys_dispatch_monthly = (resample_mys_dispatch_hourly
                                 .resample('M')
                                 .sum()
                                 .div(1e6) # MWh to TWh
                                )

# Monthly model
monthly_gen_pyspsa = (
    network
    .generators_t
    .p
    .filter(like='MYSPE')
    .groupby(by=[network.generators.carrier],axis=1)
    .sum().div(1e6) # MWh to TWh
    .resample('M')
    .sum()
    .reset_index()
    .rename(columns={'index':'timestamp'})
)

fig = make_subplots(rows=4, cols=1,shared_xaxes=False, subplot_titles = ['Coal','Gas','Hydro','Solar'])
month = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

# subplot 1 - coal generation comparison
fig.add_trace(go.Bar(x=month, y=(resample_mys_dispatch_monthly['Coal'].values-monthly_gen_pyspsa['coal'])/resample_mys_dispatch_monthly['Coal'].values*0.89, name="reference",showlegend=False,marker=dict(color='blue')),1,1)

# subplot 2 - gas generation comparison
fig.add_trace(go.Bar(x=month, y=(resample_mys_dispatch_monthly['Gas'].values-monthly_gen_pyspsa['gas'])/resample_mys_dispatch_monthly['Gas'].values*0.89, name="reference",showlegend=False,marker=dict(color='blue')),2,1)

# subplot 3 - hydro generation comparison
fig.add_trace(go.Bar(x=month, y=(resample_mys_dispatch_monthly['Hydro'].values-monthly_gen_pyspsa['hydro'])/resample_mys_dispatch_monthly['Hydro'].values*0.89, name="reference",showlegend=False,marker=dict(color='blue')),3,1)

# subplot 4 - solar generation comparison
fig.add_trace(go.Bar(x=month, y=(resample_mys_dispatch_monthly['Solar'].values-monthly_gen_pyspsa['solar'])/resample_mys_dispatch_monthly['Solar'].values*0.89, name="reference",showlegend=False,marker=dict(color='blue')),4,1)

fig.add_hline(y=0.1)
fig.add_hline(y=-0.1)


fig.update_layout(title_text="MYSPE - 2023 GENERATION COMPARISON (%)",
                  width=800,height=1000)

fig.show()



In [ ]:
# --- Calibration 5: Monthly comparison of max CF 2023 (only MYSPE) --- #

# Monthly official
cf_ofc_hourly = pd.DataFrame()
cf_ofc_hourly['Coal'] = resample_mys_dispatch_hourly['Coal'].div(1e3) / cap_mys.loc[cap_mys['Fuel'] == 'coal'].p_nom_opt.values
cf_ofc_hourly['Gas'] = resample_mys_dispatch_hourly['Gas'].div(1e3) / cap_mys.loc[cap_mys['Fuel'] == 'gas'].p_nom_opt.values
cf_ofc_hourly['Hydro'] = resample_mys_dispatch_hourly['Hydro'].div(1e3) / cap_mys.loc[cap_mys['Fuel'] == 'hydro'].p_nom_opt.values
cf_ofc_hourly['Solar'] = resample_mys_dispatch_hourly['Solar'].div(1e3) / cap_mys.loc[cap_mys['Fuel'] == 'solar'].p_nom_opt.values

cf_ofc_monthly = (cf_ofc_hourly.resample('M').max())

# Monthly model
gen_pyspsa = (
    network
    .generators_t
    .p
    .filter(like='MYSPE')
    .groupby(by=[network.generators.carrier],axis=1)
    .sum().div(1e3) # MWh to GWh
    .rename(columns={'index':'timestamp'})
)

cf_pypsa_hourly = pd.DataFrame()
cf_pypsa_hourly['coal'] = gen_pyspsa['coal'] / cap_mys.loc[cap_mys['Fuel'] == 'coal'].p_nom_opt.values
cf_pypsa_hourly['gas'] = gen_pyspsa['gas'] / cap_mys.loc[cap_mys['Fuel'] == 'gas'].p_nom_opt.values
cf_pypsa_hourly['hydro'] = gen_pyspsa['hydro'] / cap_mys.loc[cap_mys['Fuel'] == 'hydro'].p_nom_opt.values
cf_pypsa_hourly['solar'] = gen_pyspsa['solar'] / cap_mys.loc[cap_mys['Fuel'] == 'solar'].p_nom_opt.values

cf_pypsa_monthly = cf_pypsa_hourly.resample('M').max()

fig = make_subplots(rows=4, cols=1,shared_xaxes=False, subplot_titles = ['Coal','Gas','Hydro','Solar'])
month = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

# subplot 1 - coal generation comparison
fig.add_trace(go.Bar(x=month, y=cf_ofc_monthly['Coal'], name="reference",legendgroup="reference",marker=dict(color='blue')),1,1)
fig.add_trace(go.Bar(x=month, y=cf_pypsa_monthly['coal'], name="model",legendgroup="model",marker=dict(color='red')),1,1)

# subplot 2 - gas generation comparison
fig.add_trace(go.Bar(x=month, y=cf_ofc_monthly['Gas'],legendgroup="reference",showlegend=False,marker=dict(color='blue')),2,1)
fig.add_trace(go.Bar(x=month, y=cf_pypsa_monthly['gas'],legendgroup="model",showlegend=False,marker=dict(color='red')),2,1)

# subplot 3 - hydro generation comparison
fig.add_trace(go.Bar(x=month, y=cf_ofc_monthly['Hydro'],legendgroup="reference",showlegend=False,marker=dict(color='blue')),3,1)
fig.add_trace(go.Bar(x=month, y=cf_pypsa_monthly['hydro'],legendgroup="model",showlegend=False,marker=dict(color='red')),3,1)

# subplot 4 - solar generation comparison
fig.add_trace(go.Bar(x=month, y=cf_ofc_monthly['Solar'],legendgroup="reference",showlegend=False,marker=dict(color='blue')),4,1)
fig.add_trace(go.Bar(x=month, y=cf_pypsa_monthly['solar'],legendgroup="model",showlegend=False,marker=dict(color='red')),4,1)

fig.update_layout(title_text="MYSPE - 2023 CF COMPARISON",
                  width=800,height=1000)

fig.show()



In [ ]:
# --- Calibration 6: Monthly comparison of mean CF 2023 (only MYSPE) --- #

# Monthly official
cf_ofc_hourly = pd.DataFrame()
cf_ofc_hourly['Coal'] = resample_mys_dispatch_hourly['Coal'].div(1e3) / cap_mys.loc[cap_mys['Fuel'] == 'coal'].p_nom_opt.values
cf_ofc_hourly['Gas'] = resample_mys_dispatch_hourly['Gas'].div(1e3) / cap_mys.loc[cap_mys['Fuel'] == 'gas'].p_nom_opt.values
cf_ofc_hourly['Hydro'] = resample_mys_dispatch_hourly['Hydro'].div(1e3) / cap_mys.loc[cap_mys['Fuel'] == 'hydro'].p_nom_opt.values
cf_ofc_hourly['Solar'] = resample_mys_dispatch_hourly['Solar'].div(1e3) / cap_mys.loc[cap_mys['Fuel'] == 'solar'].p_nom_opt.values

cf_ofc_monthly = (cf_ofc_hourly.resample('M').min())

# Monthly model
gen_pyspsa = (
    network
    .generators_t
    .p
    .filter(like='MYSPE')
    .groupby(by=[network.generators.carrier],axis=1)
    .sum().div(1e3) # MWh to GWh
    .rename(columns={'index':'timestamp'})
)

cf_pypsa_hourly = pd.DataFrame()
cf_pypsa_hourly['coal'] = gen_pyspsa['coal'] / cap_mys.loc[cap_mys['Fuel'] == 'coal'].p_nom_opt.values
cf_pypsa_hourly['gas'] = gen_pyspsa['gas'] / cap_mys.loc[cap_mys['Fuel'] == 'gas'].p_nom_opt.values
cf_pypsa_hourly['hydro'] = gen_pyspsa['hydro'] / cap_mys.loc[cap_mys['Fuel'] == 'hydro'].p_nom_opt.values
cf_pypsa_hourly['solar'] = gen_pyspsa['solar'] / cap_mys.loc[cap_mys['Fuel'] == 'solar'].p_nom_opt.values

cf_pypsa_monthly = cf_pypsa_hourly.resample('M').min()

fig = make_subplots(rows=4, cols=1,shared_xaxes=False, subplot_titles = ['Coal','Gas','Hydro','Solar'])
month = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

# subplot 1 - coal generation comparison
fig.add_trace(go.Bar(x=month, y=cf_ofc_monthly['Coal'], name="reference",legendgroup="reference",marker=dict(color='blue')),1,1)
fig.add_trace(go.Bar(x=month, y=cf_pypsa_monthly['coal'], name="model",legendgroup="model",marker=dict(color='red')),1,1)

# subplot 2 - gas generation comparison
fig.add_trace(go.Bar(x=month, y=cf_ofc_monthly['Gas'],legendgroup="reference",showlegend=False,marker=dict(color='blue')),2,1)
fig.add_trace(go.Bar(x=month, y=cf_pypsa_monthly['gas'],legendgroup="model",showlegend=False,marker=dict(color='red')),2,1)

# subplot 3 - hydro generation comparison
fig.add_trace(go.Bar(x=month, y=cf_ofc_monthly['Hydro'],legendgroup="reference",showlegend=False,marker=dict(color='blue')),3,1)
fig.add_trace(go.Bar(x=month, y=cf_pypsa_monthly['hydro'],legendgroup="model",showlegend=False,marker=dict(color='red')),3,1)

# subplot 4 - solar generation comparison
fig.add_trace(go.Bar(x=month, y=cf_ofc_monthly['Solar'],legendgroup="reference",showlegend=False,marker=dict(color='blue')),4,1)
fig.add_trace(go.Bar(x=month, y=cf_pypsa_monthly['solar'],legendgroup="model",showlegend=False,marker=dict(color='red')),4,1)

fig.update_layout(title_text="MYSPE - 2023 CF COMPARISON (MEAN VALUE)",
                  width=1000,height=1000)

fig.show()



In [ ]:
#--- Calibration 7: Check monthly gas generation and CF (only SGP) ---#

sgp_dispatch_ofc = pd.read_csv('/home/yantidiah/tza-pypsa/SGP_hourly_demand_solargen_cost_2023-2024.csv')
sgp_dispatch_ofc['timestamp'] = pd.to_datetime(sgp_dispatch_ofc['timestamp'])
sgp_dispatch_ofc = sgp_dispatch_ofc[sgp_dispatch_ofc['timestamp'].dt.year == 2023]
sgp_dispatch_ofc.set_index('timestamp',inplace=True)

resample_sgp_dispatch_hourly = sgp_dispatch_ofc.resample('H').median(numeric_only=True)

# Monthly official generation
resample_sgp_dispatch_monthly = (resample_sgp_dispatch_hourly
                                 .resample('M')
                                 .sum()
                                 .div(1e6) # MWh to TWh
                                )

# Monthly pypsa generation
monthly_gen_pyspsa = (
    network
    .generators_t
    .p
    .filter(like='SGPXX')
    .groupby(by=[network.generators.carrier],axis=1)
    .sum().div(1e6) # MWh to TWh
    .resample('M')
    .sum()
    .reset_index()
    .rename(columns={'index':'timestamp'})
)

# Monthly official cf
cf_ofc_hourly = pd.DataFrame()
cf_ofc_hourly['Gas'] = resample_sgp_dispatch_hourly['demand_mw'].div(1e3)*0.93 / cap_sgp.loc[cap_sgp['Fuel'] == 'gas'].p_nom_opt.values
cf_ofc_hourly['Solar'] = resample_sgp_dispatch_hourly['solar_mw'].div(1e3)*0.93 / cap_sgp.loc[cap_sgp['Fuel'] == 'solar'].p_nom_opt.values
cf_ofc_monthly_mean = (cf_ofc_hourly.resample('M').mean())
cf_ofc_monthly_max = (cf_ofc_hourly.resample('M').max())

# Monthly model cf
cf_pypsa_hourly = pd.DataFrame()
cf_pypsa_hourly['gas'] = gen_pyspsa['gas'] / cap_mys.loc[cap_mys['Fuel'] == 'gas'].p_nom_opt.values
cf_pypsa_hourly['solar'] = gen_pyspsa['solar'] / cap_mys.loc[cap_mys['Fuel'] == 'solar'].p_nom_opt.values
cf_pypsa_monthly_mean = cf_pypsa_hourly.resample('M').mean()
cf_pypsa_monthly_max = cf_pypsa_hourly.resample('M').max()

fig = make_subplots(rows=3, cols=1,shared_xaxes=False, subplot_titles = ['Gas Generation (TWh) Comparison','Gas generation comparison','Gas CF Comparison (max)'])
month = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

# subplot 1 - generation comparison
fig.add_trace(go.Bar(x=month, y=resample_sgp_dispatch_monthly['demand_mw'], name="reference",legendgroup="reference",marker=dict(color='blue')),1,1)
fig.add_trace(go.Bar(x=month, y=monthly_gen_pyspsa['gas'], name="model",legendgroup="model",marker=dict(color='red')),1,1)

# # subplot 2 - cf generation comparison (mean)
fig.add_trace(go.Bar(x=month, y=(resample_sgp_dispatch_monthly['demand_mw'].values-monthly_gen_pyspsa['gas'])/resample_sgp_dispatch_monthly['demand_mw'].values, name="reference",showlegend=False,marker=dict(color='green')),2,1)

# fig.add_trace(go.Bar(x=month, y=cf_ofc_monthly_mean['Gas'], name="reference",legendgroup="reference",showlegend=False,marker=dict(color='blue')),2,1)
# fig.add_trace(go.Bar(x=month, y=cf_pypsa_monthly_mean['gas'], name="model",legendgroup="model",showlegend=False,marker=dict(color='red')),2,1)

# subplot 3 - cf generation comparison (max)
fig.add_trace(go.Bar(x=month, y=cf_ofc_monthly_max['Gas'], name="reference",legendgroup="reference",showlegend=False,marker=dict(color='blue')),3,1)
fig.add_trace(go.Bar(x=month, y=cf_pypsa_monthly_max['gas'], name="model",legendgroup="model",showlegend=False,marker=dict(color='red')),3,1)

fig.update_layout(title_text="SGPXX - 2023 GENERATION AND MEAN CF COMPARISON",
                  width=1000,height=800)

fig.show()


In [ ]:
network.statistics.expanded_capacity()